In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 22, Local frames and frame decomposition

Companion notebook to [22_frame_decomposition.md](22_frame_decomposition.md). `LocalFrame` is a library wrapper (not an Expr) bundling a frame's identity and display symbols. It produces frame VFs `X_a`, dual coframes `e^a`, and the frame-scoped duality rule. Frame-decomposition rules are opt-in to avoid loops.

## `LocalFrame`, the wrapper

Frames with the same `(name, dim, vf_symbol, coframe_symbol)` compare equal. `dim=None` is the symbolic-dimension mode Faz 17 proofs use.

In [ ]:
from jacopy.calculus.local_frame import LocalFrame, FrameIndex

F = LocalFrame("F", dim=3)
print(f"frame    : {F}")
print(f"name     : {F.name}")
print(f"dim      : {F.dim}")
print(f"vf sym   : {F.vf_symbol}")
print(f"coframe  : {F.coframe_symbol}")

## Indices and basis elements

`FrameVectorField` subclasses `Derivation` (degree 0), every existing pass picks frame VFs up automatically. `FrameCovector` is an `Atom` mediated through `Pairing`. Equality includes the frame name to keep coexisting frames distinct.

In [ ]:
a, b = F.index("a"), F.index("b")
X_a, X_b = F.X(a),       F.X(b)
e_a, e_b = F.coframe(a), F.coframe(b)

print(f"X_a : {X_a}    class={X_a.__class__.__name__}")
print(f"e_a : {e_a}    class={e_a.__class__.__name__}")

## `KroneckerDelta`, the contraction unit

`KroneckerDelta(i, j)` collapses to `One` when the indices are structurally equal; stays opaque otherwise.

In [ ]:
from jacopy.calculus.local_frame import KroneckerDelta

print(f"δ^a_a = {KroneckerDelta(a, a)}")
print(f"δ^a_b = {KroneckerDelta(a, b)}")

## `FramePairingDualityDefinition`, `⟨e^a, X_b⟩ → δ^a_b`

Frame-scoped: fires only when both halves belong to the same `LocalFrame`. Build via `F.duality_definition()` to wire the scoping correctly.

In [ ]:
from jacopy.calculus.pairing import Pairing
from jacopy.proof.expansion import ExpansionEngine

p = Pairing(e_b, X_a)
print(f"raw pairing : {p}")

engine = ExpansionEngine([F.duality_definition()])
out, steps = engine.expand(p)
print(f"after rule  : {out}")
print(f"rule        : {steps[0].rule}")

## Frame decomposition, opt-in rules

`FrameDecompositionDefinition(F)` rewrites `W → Σ_a e^a(W) · X_a`. **Opt-in**: pairing it with duality creates a loop. `CartanStructureProblem` (tutorial 23) turns it on for a specific sub-pass.

In [ ]:
from jacopy.algebra.derivation import Derivation
from jacopy.calculus.frame_decomposition import FrameDecompositionDefinition

W = Derivation("W", 0)
rule = FrameDecompositionDefinition(F)

# Apply once directly (running the engine would loop).
print(f"W → {rule.rewrite(W)}")

## The three frame-decomposition rules

| Rule | Folds |
|---|---|
| `FrameDecompositionDefinition(F)` | `W → Σ_a e^a(W) · X_a` for any non-frame VF |
| `ConnectionEvalYFrameDecompositionDefinition(F, ∇)` | `∇_X Y → ∇_X (Σ_a e^a(Y) · X_a)` |
| `ConnectionFormDecompositionDefinition(F, ∇, ω)` | `∇_V X_b → Σ_c ω^c_b(V) · X_c` |

The third introduces the **connection form** `ω^c_b(∇)`, the keystone of Cartan structure equation proofs. Every Christoffel-symbol calculation funnels through it.

## Summary

* `LocalFrame` library wrapper; `dim=None` for symbolic dimension. Equality on `(name, dim, vf_symbol, coframe_symbol)` tuples.
* Four Expr shapes: `FrameIndex`, `FrameVectorField` (Derivation subclass), `FrameCovector` (Atom), `KroneckerDelta` (collapses on matching indices).
* `FramePairingDualityDefinition`, frame-scoped duality rule. Build via `F.duality_definition()`.
* Three opt-in frame-decomposition rules; `ConnectionFormDecompositionDefinition` introduces `ω^c_b`. `CartanStructureProblem` (tutorial 23) handles the loop-avoidance bookkeeping.